Assess how well top-ranked ligands can predict a gene set of interest
================

This tutorial assesses the ligands prioritized by NicheNet in their
ability to predict a gene set of interest. We will first follow the
steps of [Perform NicheNet analysis starting from an AnnData
object](wrapper.ipynb)) to obtain ligands rankings. Make sure you
understand the steps and output of a basic NicheNet analysis (more
information in [Perform NicheNet analysis starting from an AnnData object:
step-by-step analysis](steps.ipynb). You can also apply this
tutorial to the [NicheNet’s ligand activity analysis on a gene set of
interest](ligand_activity_gene_set.ipynb) notebook.


In [1]:
from nichenetpy.wrappers import (
    run_nichenet,
    calculate_fraction_top_predicted,
    calculate_fraction_top_predicted_fisher
)
from nichenetpy.gene_symbol import mouse_alias_info
from nichenetpy.prediction import assess_rf_class_probabilities
from nichenetpy.metrics import calculate_metrics

import anndata
import os
import requests
import pickle
import pandas as pd

Download the model pickle

In [2]:
filename = "nichenet_mouse.pkl"
file_path = os.path.join("./tutorial_files", filename)
if not os.path.exists(file_path):
    res = requests.get(f"https://zenodo.org/records/14887637/files/{filename}")
    with open(file_path, "wb") as file:
        file.write(res.content)

Download the AnnData object

In [3]:
data_path = os.path.normpath("./tutorial_files/AnnData")
if not os.path.exists(data_path):
    os.makedirs(data_path)
filename = "annData3531889.h5"
file_path = os.path.join(data_path, filename)
if not os.path.exists(file_path):
    res = requests.get(f"https://zenodo.org/records/14859451/files/{filename}")
    with open(file_path, "wb") as file:
        file.write(res.content)

Perform nichenet analysis using the wrapper function

In [4]:
ann = anndata.io.read_h5ad(os.path.join(data_path, "annData3531889.h5"))
ann.var_names = ann.var["gene"]
mouse_alias_info.alias_to_symbol(ann)
with open("./tutorial_files/nichenet_mouse.pkl", "rb") as file:
    model = pickle.loads(file.read())
predictor = model["predictor"]
lr_network = model["lr_network"]
lr_sig = model["lr_sig"]
receiver = "CD8 T"
sender_celltypes = ["CD4 T","Treg", "Mono", "NK", "B", "DC"]
output = run_nichenet(
    ann,
    predictor,
    lr_network,
    lr_sig=lr_sig,
    receiver=receiver,
    sender_celltypes=sender_celltypes,
    condition_col="aggregate",
    condition_oi="LCMV",
    condition_ref="SS",
    get_expressed_genes_pct=0.05
)
geneset_oi = output["geneset_oi"]
expressed_genes_receiver = output["expressed_genes_receiver"]
ligands_oi = output["best_upstream_ligands"]

## Assess how well top-ranked ligands can predict a gene set of interest

For the top 30 ligands, we will now build a multi-ligand model that uses
all top-ranked ligands to predict whether a gene belongs to the gene set
of interest (differentially expressed genes in CD8 T cells after LCMV
infection) or not. This classification model will be trained via
cross-validation and returns a probability for every gene.

In [5]:
n = 2
k = 3
gene_predictions_top30_list = [
    assess_rf_class_probabilities(
        folds=k,
        geneset=geneset_oi,
        background_expressed_genes=expressed_genes_receiver,
        ligands_oi=ligands_oi,
        predictor=predictor
    ) for _ in range(n)
]

Evaluate how well the target gene probabilities accord to the gene set assignments.

In [6]:
target_prediction_performances = []
for df in gene_predictions_top30_list:
    met = calculate_metrics(list(df["prediction"]), list(df["response"]))
    target_prediction_performances.append(pd.DataFrame([list(met.values())], columns=met.keys()))
target_prediction_performances = pd.concat(target_prediction_performances)
target_prediction_performances.reset_index(drop=True, inplace=True)

What is the AUROC, AUPR and PCC of this model (averaged over cross-validation rounds)?

In [7]:
target_prediction_performances.mean()

auroc             0.814122
pearson           0.548842
aupr              0.485329
aupr_corrected    0.409804
dtype: float64

Evaluate whether genes belonging to the gene set are more likely to be top-predicted. We will look at the top 5% of predicted targets here.

In [8]:
target_prediction_performances_discrete = pd.concat([calculate_fraction_top_predicted(df) for df in gene_predictions_top30_list])
target_prediction_performances_discrete

,true_target,n,positive_prediction,fraction_positive_predicted
0,0,2950,53,0.017966
1,1,241,108,0.448133
0,0,2950,47,0.015932
1,1,241,114,0.473029


What is the fraction of viral response genes that belongs to the top 5% predicted targets?

In [9]:
target_prediction_performances_discrete[
    target_prediction_performances_discrete["true_target"] == 1
]["fraction_positive_predicted"].mean()

np.float64(0.46058091286307057)

What is the fraction of non-viral-response genes that belongs to the top 5% predicted targets?

In [10]:
target_prediction_performances_discrete[
    target_prediction_performances_discrete["true_target"] == 0
]["fraction_positive_predicted"].mean()

np.float64(0.01694915254237288)

We see that the viral response genes are enriched in the top-predicted target genes. To test this, we will now apply a Fisher’s exact test for every cross-validation round and report the average p-value.

In [13]:
target_prediction_performances_discrete_fisher = [
    calculate_fraction_top_predicted_fisher(e).pvalue for e in gene_predictions_top30_list
]
sum(target_prediction_performances_discrete_fisher)/len(target_prediction_performances_discrete_fisher)

np.float64(1.0055556476075937e-91)

Finally, we will look at which p-EMT genes are well-predicted in every cross-validation round.